## Exercises on Policy Evaluation (Model-Free Prediction)

These paper-and-pencil exercises reinforce Chapter 05: Monte Carlo prediction (first-visit vs every-visit), one-step Temporal-Difference learning and bootstrapping, the $n$-step return, the batch MC-vs-TD distinction (certainty equivalence), and the $\lambda$-return. Notation: $G_{t}$ is the return, $G_{t:t+n}$ the $n$-step return, $V(s)$ the learned estimate of $v_\pi(s)$; the reward for acting at $t$ is $R_{t+1}$.

### Exercise 5.1 — First-visit vs every-visit Monte Carlo

A single episode produces the trajectory (states and the reward received on each transition), with **$\gamma=0.9$**:

$\displaystyle S_0=C \xrightarrow{R_1=0} S_1=B \xrightarrow{R_2=0} S_2=C \xrightarrow{R_3=0} S_3=D \xrightarrow{R_4=1} \text{terminal}.$

State $C$ is visited **twice** (at $t=0$ and $t=2$). Using Monte Carlo prediction, estimate $V(C)$ by both the **first-visit** and **every-visit** methods, and also give $V(B)$ and $V(D)$.

**Step 1 — Compute the return from each visited time step**, $G_t=\sum_{k\ge0}\gamma^{k}R_{t+k+1}$ (with $\gamma=0.9$, so $\gamma^2=0.81,\ \gamma^3=0.729$):

$\displaystyle G_0\,(C) = 0 + 0.9(0) + 0.81(0) + 0.729(1) = 0.729,$
$\displaystyle G_1\,(B) = 0 + 0.9(0) + 0.81(1) = 0.81,$
$\displaystyle G_2\,(C) = 0 + 0.9(1) = 0.9,$
$\displaystyle G_3\,(D) = 1 = 1.$

**Step 2 — First-visit MC.** Use only the return following the *first* visit to each state. For $C$ the first visit is at $t=0$:

$\displaystyle V_{\text{FV}}(C) = G_0 = 0.729.$

**Step 3 — Every-visit MC.** Average the returns of *all* visits to $C$ (at $t=0$ and $t=2$):

$\displaystyle V_{\text{EV}}(C) = \frac{G_0 + G_2}{2} = \frac{0.729 + 0.9}{2} = 0.8145.$

The other states are visited once, so both methods agree: $V(B)=0.81,\ V(D)=1.$

**Step 4 — Interpret the difference.** First-visit and every-visit give **different** estimates for $C$ ($0.729$ vs $0.8145$) because the second visit is closer to the reward and thus has a larger discounted return. Both are unbiased and converge to $v_\pi(C)$ as the number of episodes grows; they differ only in how a single episode's repeated visits are counted.

**Key concept**

Monte Carlo estimates a state's value as the **empirical mean of the actual returns** that followed it — no model, no bootstrapping. First-visit and every-visit are two consistent bookkeeping choices with the same limit.

### Exercise 5.2 — One episode of TD(0)

In the 5-state Random Walk (states $A,B,C,D,E$ with terminal states on both ends), all estimates are initialised to $V=0.5$, the terminal states have value $0$, $\gamma=1$, and the step size is $\alpha=0.1$. The agent experiences the episode

$\displaystyle C \xrightarrow{R=0} D \xrightarrow{R=0} E \xrightarrow{R=1} \text{terminal (right)}.$

Apply the TD(0) update $V(S_t)\leftarrow V(S_t)+\alpha\,[\,R_{t+1}+\gamma V(S_{t+1})-V(S_t)\,]$ at each step and report the resulting values.

**Step 1 — Transition $C\to D$.** TD error $\delta = R+\gamma V(D)-V(C) = 0 + 1(0.5) - 0.5 = 0.$
$\displaystyle V(C) \leftarrow 0.5 + 0.1(0) = 0.5 \quad(\text{unchanged}).$

**Step 2 — Transition $D\to E$.** $\delta = 0 + 0.5 - 0.5 = 0.$
$\displaystyle V(D) \leftarrow 0.5 + 0.1(0) = 0.5 \quad(\text{unchanged}).$

**Step 3 — Transition $E\to\text{terminal}$.** The terminal value is $0$, so $\delta = R + \gamma\cdot 0 - V(E) = 1 + 0 - 0.5 = 0.5.$
$\displaystyle V(E) \leftarrow 0.5 + 0.1(0.5) = 0.55.$

**Step 4 — Summary.** After this episode only $V(E)$ changes, to $0.55$; all other estimates are unchanged.

**Step 5 — Why.** With every interior estimate equal to $0.5$, each interior TD error is $0$ (the bootstrap target equals the current value). Only the transition that actually *reaches the reward* injects new information — and it does so **immediately**, without waiting for the episode's end, unlike Monte Carlo.

**Key concept**

TD(0) **bootstraps**: it updates $V(S_t)$ toward $R_{t+1}+\gamma V(S_{t+1})$, an *estimate* of the return. Information about the reward propagates one state back per episode, so it can learn online and from incomplete episodes.

### Exercise 5.3 — The $n$-step return

Along a trajectory the rewards from $t=0$ are $R_1,\dots,R_5 = 0,0,0,0,1$, with $\gamma=0.9$. The current value estimates of the states reached are $V(S_1)=0.4,\ V(S_2)=0.5,\ V(S_3)=0.6$. Compute the $1$-step, $2$-step and $3$-step returns $G_{0:1}, G_{0:2}, G_{0:3}$, and the full Monte Carlo return $G_0$. Comment on the bias–variance trade-off.

**Step 1 — Definition.** The $n$-step return sums $n$ real rewards and then **bootstraps** on the value of the state reached:

$\displaystyle G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{\,n-1}R_{t+n} + \gamma^{\,n} V(S_{t+n}).$

**Step 2 — Evaluate** (all early rewards are $0$; $\gamma^2=0.81,\ \gamma^3=0.729,\ \gamma^4=0.6561$):

$\displaystyle G_{0:1} = R_1 + \gamma V(S_1) = 0 + 0.9(0.4) = 0.36,$
$\displaystyle G_{0:2} = R_1 + \gamma R_2 + \gamma^2 V(S_2) = 0 + 0 + 0.81(0.5) = 0.405,$
$\displaystyle G_{0:3} = R_1 + \gamma R_2 + \gamma^2 R_3 + \gamma^3 V(S_3) = 0 + 0 + 0 + 0.729(0.6) = 0.4374.$

**Step 3 — Monte Carlo (full) return.** No bootstrapping; only the terminal reward $R_5=1$ survives, discounted by $\gamma^4$:

$\displaystyle G_0 = \gamma^4 (1) = 0.6561.$

**Step 4 — Interpret.** As $n$ increases, the estimate relies on **more real reward and less bootstrap** ($0.36 \to 0.405 \to 0.4374 \to 0.6561$). Small $n$ (TD-like) has **low variance but higher bias** (it leans on possibly-wrong value estimates); large $n$ (MC-like) has **low bias but higher variance** (it accumulates the randomness of many steps). Intermediate $n$ balances the two.

**Key concept**

The $n$-step return interpolates between TD(0) ($n=1$) and Monte Carlo ($n=\infty$), letting us **tune how much we bootstrap** and thereby trade bias against variance.

### Exercise 5.4 — Batch MC vs TD: the A/B example

You observe the following **eight** episodes (state, reward, …, termination), with $\gamma=1$:

- Episode 1: $A,\,0,\,B,\,0$ (start in $A$, reward $0$, go to $B$, reward $0$, terminate)
- Episodes 2–7: $B,\,1$ (start in $B$, reward $1$, terminate) — six such episodes
- Episode 8: $B,\,0$ (start in $B$, reward $0$, terminate)

Under **batch** updating (repeatedly replay these episodes until convergence), compute the estimates $V(A)$ and $V(B)$ produced by (i) batch Monte Carlo and (ii) batch TD(0). Explain why they differ.

**Step 1 — Estimate $V(B)$.** State $B$ terminates immediately with reward $1$ in six of the eight episodes it appears in, and $0$ in the other two:

$\displaystyle V(B) = \frac{6\times 1 + 2\times 0}{8} = \frac{6}{8} = 0.75.$

Both methods agree on $V(B)$.

**Step 2 — Batch Monte Carlo estimate of $V(A)$.** $A$ occurs exactly once (Episode 1), and the **actual return** that followed it was $0+0=0$. MC minimises mean-squared error on the observed returns, so

$\displaystyle V_{\text{MC}}(A) = 0.$

**Step 3 — Batch TD(0) estimate of $V(A)$.** TD builds the **maximum-likelihood (certainty-equivalence) model** from the data: from $A$ the process went to $B$ $100\%$ of the time with reward $0$. Hence

$\displaystyle V_{\text{TD}}(A) = r(A) + \gamma\,V(B) = 0 + 1\times 0.75 = 0.75.$

**Step 4 — Why they differ.** MC answers "what return actually followed $A$ in the data?" ($0$, from its single sample). TD answers "given the estimated Markov model, what is the value of $A$?" — and since $A$ always leads to $B$, and $B$ is worth $0.75$, $A$ should also be worth $0.75$. MC minimises error on the *training data*; TD is exactly right *if the Markov model is right*, and therefore usually generalises better to new data.

**Key concept**

Batch TD converges to the **certainty-equivalence** estimate (optimal for the maximum-likelihood MDP), while batch MC converges to the minimum-training-error estimate. This is the deep reason TD often learns faster than MC on Markov problems.

### Exercise 5.5 — The $\lambda$-return

For a 3-step episode terminating after $R_3$, with $\gamma=1$ and $\lambda=0.5$, the rewards are $R_1=0,\ R_2=0,\ R_3=1$ and the intermediate value estimates are $V(S_1)=0.5,\ V(S_2)=0.5$. Compute the forward-view $\lambda$-return $G_0^\lambda$, and check the two limiting cases $\lambda=0$ and $\lambda=1$.

**Step 1 — The component $n$-step returns** (episode ends after 3 steps, so $T-t-1=2$ bootstrapped terms plus the full return):

$\displaystyle G_{0:1} = R_1 + \gamma V(S_1) = 0 + 0.5 = 0.5,$
$\displaystyle G_{0:2} = R_1 + \gamma R_2 + \gamma^2 V(S_2) = 0 + 0 + 0.5 = 0.5,$
$\displaystyle G_{0:3} = R_1 + R_2 + R_3 = 1 \quad(\text{the full return } G_0).$

**Step 2 — Weighted combination.** The $\lambda$-return weights the $n$-step returns by $(1-\lambda)\lambda^{n-1}$, with the remaining weight $\lambda^{T-t-1}$ on the full return:

$\displaystyle G_0^\lambda = (1-\lambda)\big[\lambda^0 G_{0:1} + \lambda^1 G_{0:2}\big] + \lambda^2 G_{0:3}.$

**Step 3 — Substitute** $\lambda=0.5$:

$\displaystyle G_0^\lambda = 0.5\big[(1)(0.5) + (0.5)(0.5)\big] + (0.25)(1) = 0.5(0.75) + 0.25 = 0.625.$

(Weight check: $(1-\lambda)(1+\lambda)+\lambda^2 = 1-\lambda^2+\lambda^2 = 1$ ✓.)

**Step 4 — Limiting cases.**
- $\lambda=0$: all weight on $G_{0:1}=0.5$ → recovers **TD(0)**.
- $\lambda=1$: all weight on $G_{0:3}=1$ → recovers **Monte Carlo**.

**Key concept**

The $\lambda$-return is a single target that **geometrically averages all $n$-step returns**, smoothly interpolating between TD(0) ($\lambda=0$) and MC ($\lambda=1$). In practice it is realised online via eligibility traces (backward view).